# NVIDIA Nemotron 3.5 Super VL: customer-service returns agent

This runnable Microsoft Foundry demo uses a fictional Northline Audio return case. NVIDIA Nemotron 3.5 Super VL reads an image of an order record with an item table, shipment events, and a customer return request. A Foundry agent then searches two company documents: the returns policy and the Aurora H7 support bulletin. It must connect the visual facts with both sources before advising the customer.

The scenario implements the ideas in the [NVIDIA specialized-agents article](https://developer.nvidia.com/blog/nvidia-nemotron-3-5-super-vl-powers-specialized-agents-with-multimodal-reasoning/): a customer-service agent follows return policy; a research agent connects evidence across sources; a document agent interprets tables and page layout. All customer, order, and policy data here are synthetic.

| Article concept | This demo |
| --- | --- |
| Customer service follows policy | The agent applies the 30-day standard and 45-day suspected-defect rules without approving an RMA itself. |
| Research connects sources | Foundry file search brings together the returns policy and product support bulletin. |
| Document intelligence interprets layout | Nemotron reads the item table, shipment-event table, and return-request panel in one image. |

**Architecture:** the Foundry agent requests a local read_order_document function. Python sends the single PNG to the working Chat Completions endpoint and returns visual facts. Foundry file search retrieves the policy and support bulletin from a vector store. The agent, conversation, tool calls, and traces remain available for inspection. Direct image input to this deployment through Agent Service Responses returned HTTP 400 in the earlier demo, so this notebook uses the proven function route.


## Three-minute presenter script

| Time | Show | Say |
| --- | --- | --- |
| 0:00–0:30 | Order image | "This is one visual document. The model must read the item table, delivery event, and customer's report." |
| 0:30–1:15 | Question 1 and function call | "Foundry asks our Python function to read the image with NVIDIA Nemotron 3.5 Super VL. The function returns observed facts, not a policy decision." |
| 1:15–2:15 | Question 2 and file search | "The 30-day standard window and 45-day defect review come from the returns policy. The required checks come from a separate product bulletin." |
| 2:15–3:00 | Customer draft and traces | "The answer is conditional: support still has to verify the issue. We can inspect the saved conversation and agent traces." |

**Expected decision as of 2026-10-21:** delivery was 2026-09-18, so the request is 33 days later. The 30-day standard window has passed; the 45-day suspected-defect review remains open through 2026-11-02. A customer-reported defect is not yet a confirmed defect or approved refund.


## 0. Setup

Follow [the sample README](README.md) to install dependencies, sign in with Microsoft Entra, and set the Foundry project endpoint and model deployment. Run cells in order. This notebook creates or reuses the two-document vector store by content hash. It leaves its agent, vector store, and conversations in Foundry for inspection.


In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path

from IPython.display import Image, Markdown, display
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FileSearchTool, FunctionTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI


In [ ]:
PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "").rstrip("/")
MODEL = os.environ.get("FOUNDRY_MODEL", "")
if not PROJECT_ENDPOINT or not MODEL:
    raise EnvironmentError("Set FOUNDRY_PROJECT_ENDPOINT and FOUNDRY_MODEL before running this notebook.")
INFERENCE_BASE_URL = os.getenv(
    "FOUNDRY_OPENAI_BASE_URL",
    PROJECT_ENDPOINT.rstrip("/") + "/openai/v1/",
)
AGENT_NAME = os.getenv("FOUNDRY_CUSTOMER_SERVICE_AGENT_NAME", "nemotron-customer-service-returns")
KNOWLEDGE_STORE_NAME = "Northline Audio customer-service knowledge"
asset_candidates = [
    Path.cwd() / "customer_service_assets",
    Path.cwd() / "inference" / "msfoundry" / "nemotron-35-vl-super" / "customer_service_assets",
]
ASSET_DIR = next((path for path in asset_candidates if (path / "northline_order_return_case.png").is_file()), None)
if ASSET_DIR is None:
    raise FileNotFoundError("Open this notebook from its directory or the repository root.")
IMAGE_PATH = ASSET_DIR / "northline_order_return_case.png"
KNOWLEDGE_PATHS = [
    ASSET_DIR / "northline_returns_policy.txt",
    ASSET_DIR / "aurora_h7_support_bulletin.txt",
]
if not all(path.is_file() for path in KNOWLEDGE_PATHS):
    raise FileNotFoundError("The returns policy and support bulletin must be beside the image.")
if not INFERENCE_BASE_URL.rstrip("/").endswith("/openai/v1"):
    raise ValueError("FOUNDRY_OPENAI_BASE_URL must end in /openai/v1/")
# Set up identity here so later cells work even if the optional token check is skipped.
credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
print(f"Project: {PROJECT_ENDPOINT}")
print(f"Agent: {AGENT_NAME}")
print(f"Model deployment: {MODEL}")
display(Image(filename=str(IMAGE_PATH), width=850))


## 1. Authenticate with Microsoft Entra

The configuration cell creates the reusable credential and token provider. This cell confirms that the identity can get a Foundry token; it is safe to rerun. Locally, DefaultAzureCredential uses az login; an Azure-hosted app can use managed identity. No API key is stored in this notebook.


In [ ]:
# Optional early check: configuration already created the reusable identity.
credential = globals().get("credential") or DefaultAzureCredential(
    exclude_interactive_browser_credential=True
)
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
credential.get_token("https://ai.azure.com/.default")
print("Microsoft Entra identity is ready.")


## 2. Define the visual-document function

The function asks NVIDIA Nemotron 3.5 Super VL to transcribe the visible document sections. It does not decide eligibility. The customer request is a reported claim, not proof that the defect or returned contents have been verified.


In [ ]:
def read_order_document() -> str:
    image_url = "data:image/png;base64," + base64.b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
    with OpenAI(api_key=token_provider, base_url=INFERENCE_BASE_URL, timeout=120.0) as vision_client:
        response = vision_client.chat.completions.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": (
                        "Read the attached order and return-case document. Transcribe the order ID, "
                        "product, SKU, price, order date, delivery date from the shipment table, "
                        "case review date, return-request date, reported issue, reported troubleshooting, "
                        "and listed box contents. Distinguish recorded order/shipment facts from "
                        "the customer's unverified statements. Do not apply any return policy. "
                        "Mark any unreadable field as unknown."
                    )},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            }],
        )
    facts = response.choices[0].message.content
    if not facts:
        raise RuntimeError("NVIDIA Nemotron 3.5 Super VL returned no document facts.")
    return facts


## 3. Reuse or index the two company sources

File search gives the agent two independent sources. The returns policy controls windows, shipping, fees, and refund timing; the product bulletin controls troubleshooting. This cell selects the existing Northline vector store even when it is only partially populated, compares **SHA-256 hashes of indexed file content** with the local documents, and uploads only missing content. If it finds duplicate, failed, or changed copies, it stops for inspection instead of adding another file. Re-running the cell with the same documents makes no index changes.

In [ ]:
# Repair a kernel where the earlier authentication cell was skipped.
if "credential" not in globals():
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
if "token_provider" not in globals():
    token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")

local_documents = {path.name: (path, hashlib.sha256(path.read_bytes()).hexdigest())
                   for path in KNOWLEDGE_PATHS}
if len({digest for _, digest in local_documents.values()}) != len(local_documents):
    raise RuntimeError("The two local knowledge documents have identical content.")

project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
knowledge_client = None
try:
    knowledge_client = project.get_openai_client()
    requested_store_id = os.getenv("FOUNDRY_CUSTOMER_SERVICE_VECTOR_STORE_ID")
    if requested_store_id:
        vector_store = knowledge_client.vector_stores.retrieve(requested_store_id)
    else:
        named_stores = [store for store in knowledge_client.vector_stores.list()
                        if store.name == KNOWLEDGE_STORE_NAME]
        if len(named_stores) > 1:
            raise RuntimeError(
                f"Several stores are named {KNOWLEDGE_STORE_NAME!r}: "
                f"{[store.id for store in named_stores]}. Set "
                "FOUNDRY_CUSTOMER_SERVICE_VECTOR_STORE_ID to select one."
            )
        vector_store = named_stores[0] if named_stores else None
    if vector_store is None:
        vector_store = knowledge_client.vector_stores.create(name=KNOWLEDGE_STORE_NAME)
        print(f"Created vector store {vector_store.id}")

    def indexed_documents():
        entries = []
        for attachment in knowledge_client.vector_stores.files.list(vector_store.id):
            metadata = knowledge_client.files.retrieve(attachment.id)
            content = knowledge_client.files.content(attachment.id).read()
            entries.append({
                "id": attachment.id,
                "name": metadata.filename,
                "status": attachment.status,
                "sha256": hashlib.sha256(content).hexdigest(),
            })
        return entries

    entries = indexed_documents()
    missing = []
    for name, (path, digest) in local_documents.items():
        same_content = [entry for entry in entries if entry["sha256"] == digest]
        same_name_other_content = [entry for entry in entries
                                   if entry["name"] == name and entry["sha256"] != digest]
        if same_name_other_content:
            raise RuntimeError(f"{name} is already indexed with different content; review before replacing it.")
        if len(same_content) > 1:
            raise RuntimeError(f"Duplicate indexed content for {name}: {[e['id'] for e in same_content]}")
        if same_content and same_content[0]["status"] != "completed":
            raise RuntimeError(f"Existing copy of {name} is {same_content[0]['status']}; review it before retrying.")
        if same_content:
            print(f"Reusing {name}: {same_content[0]['id']}")
        else:
            missing.append(path)

    for path in missing:
        indexed = knowledge_client.vector_stores.files.upload_and_poll(
            vector_store_id=vector_store.id,
            file=(path.name, path.read_bytes(), "text/plain"),
        )
        if indexed.status != "completed":
            raise RuntimeError(f"Indexing failed for {path.name}: {indexed.last_error}")
        print(f"Indexed missing document {path.name}: {indexed.id}")

    final_entries = indexed_documents()
    for name, (_, digest) in local_documents.items():
        matches = [entry for entry in final_entries
                   if entry["sha256"] == digest and entry["status"] == "completed"]
        if len(matches) != 1:
            raise RuntimeError(f"Expected exactly one completed indexed copy of {name}; found {len(matches)}.")
    VECTOR_STORE_ID = vector_store.id
    print(f"Foundry vector store: {VECTOR_STORE_ID}")
    print(f"Completed indexed files: {sum(e['status'] == 'completed' for e in final_entries)}")
finally:
    if knowledge_client is not None:
        knowledge_client.close()
    project.close()

## 4. Create or update the Foundry agent

The agent has a local visual-document function and Foundry file search. It is instructed to separate observed order facts, customer statements, policy rules, and product guidance. A new version is created so the result remains visible in Foundry after the demo.


In [ ]:
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
try:
    visual_tool = FunctionTool(
        name="read_order_document",
        description="Read the customer-supplied order and return-case image; return only visible document facts.",
        parameters={"type": "object", "properties": {}, "required": [], "additionalProperties": False},
        strict=True,
    )
    knowledge_tool = FileSearchTool(vector_store_ids=[VECTOR_STORE_ID])
    agent = project.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=MODEL,
            instructions=(
                "You are Northline Audio's customer-service returns specialist in a fictional case. "
                "For order, shipment, item, and customer-statement facts, call read_order_document "
                "unless those visual facts are already in the conversation. For policy, dates, "
                "shipping, fees, refund timing, troubleshooting, or escalation, use file search "
                "to consult both the Northline Audio Returns Policy and Aurora H7 No-Audio Support Bulletin. "
                "The case review date is 2026-10-21. Count calendar days from the day after delivery. "
                "Separate visual facts, unverified customer statements, policy rules, and support guidance. "
                "For nonvisual claims, name the source title in plain text in parentheses, "
                "for example (Northline Audio Returns Policy) or "
                "(Aurora H7 No-Audio Support Bulletin). Do not rely on citation markers alone. "
                "Do not say a defect is confirmed, an RMA is issued, or a refund is approved. "
                "Only explain conditional eligibility and the steps a human support team must complete."
            ),
            tools=[visual_tool, knowledge_tool],
        ),
    )
    print(f"Created Foundry agent {agent.name} (version {agent.version})")
finally:
    project.close()


## 5. Define the question runner

The runner fulfills read_order_document with a local Chat Completions call. Foundry executes file search server-side. For a knowledge question it requests file search and verifies either a new call or a completed search earlier in the same conversation; repeated questions may reuse retrieved context. It prints response IDs and a portal Traces link.


In [ ]:
def ask_service_agent(
    question: str,
    conversation_id: str | None = None,
    require_visual_tool: bool = False,
    require_file_search: bool = False,
) -> str:
    project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
    agent_client = None
    try:
        agent_client = project.get_openai_client(agent_name=agent.name)
        if conversation_id is None:
            conversation_id = agent_client.conversations.create().id
            print(f"New Foundry conversation: {conversation_id}")
        else:
            print(f"Continuing Foundry conversation: {conversation_id}")
        print(f"Question: {question}")
        prior_file_search_calls = 0
        if require_file_search:
            prior_file_search_calls = sum(
                item.type == "file_search_call" and item.status == "completed"
                for item in agent_client.conversations.items.list(conversation_id, order="asc")
            )
        request = {"conversation": conversation_id, "input": question}
        if require_file_search:
            request["tool_choice"] = {"type": "file_search"}
        response = agent_client.responses.create(**request)
        file_search_calls = 0
        visual_calls = 0
        visual_facts = None
        for _ in range(4):
            file_search_calls += sum(item.type == "file_search_call" for item in response.output)
            calls = [item for item in response.output if item.type == "function_call"]
            if not calls:
                if not response.output_text:
                    raise RuntimeError("Agent returned no final text.")
                break
            if any(item.name != "read_order_document" for item in calls):
                raise RuntimeError(f"Unexpected function call: {[item.name for item in calls]}")
            visual_calls += len(calls)
            print(f"Agent requested {len(calls)} visual-document function call(s).")
            if visual_facts is None:
                visual_facts = read_order_document()
                print("Visual facts returned by NVIDIA Nemotron 3.5 Super VL:")
                print(visual_facts)
            tool_outputs = [
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps({"document_facts": visual_facts}),
                }
                for item in calls
            ]
            response = agent_client.responses.create(conversation=conversation_id, input=tool_outputs)
        else:
            raise RuntimeError("Agent exceeded the visual-tool round limit.")
        print(f"Visual function calls in this response: {visual_calls}")
        print(f"Foundry file-search calls in this response: {file_search_calls}")
        if require_visual_tool and visual_calls == 0:
            raise RuntimeError("The agent did not request the visual-document function.")
        if require_file_search and file_search_calls == 0 and prior_file_search_calls == 0:
            print(f"Conversation: {conversation_id}; response: {response.id}")
            print("Answer without file-search evidence:", response.output_text)
            raise RuntimeError("No completed Foundry file search exists in this conversation.")
        if require_file_search and file_search_calls == 0:
            print(f"Reused {prior_file_search_calls} earlier completed file-search call(s).")
        print(f"Final response ID (search in Traces): {response.id}")
        print("Open https://ai.azure.com/ to inspect this agent and search for the response ID in Traces.")
        display(Markdown("### Customer-service agent answer\n\n" + response.output_text))
        return conversation_id
    finally:
        if agent_client is not None:
            agent_client.close()
        project.close()


## 6. Question 1 — interpret the visual document

This starts a new saved conversation. The agent must call the local visual function to read the order table, shipment event, and customer's reported issue.


In [ ]:
customer_conversation = ask_service_agent(
    "Read the attached Northline order and return-case document. What was ordered, "
    "when was it delivered, when was the return request submitted, and what does the "
    "customer say happened? Separate recorded facts from unverified statements.",
    require_visual_tool=True,
)


## 7. Question 2 — connect the image with two knowledge sources

Use the same conversation. The policy controls the return window and refund conditions; the support bulletin describes the remaining troubleshooting. The runner verifies that Foundry searched the knowledge documents at least once.


In [ ]:
customer_conversation = ask_service_agent(
    "As of the case review date, is this request within the standard-return window "
    "or the suspected-defect review window? Use the recorded delivery date from the "
    "image, calculate the days elapsed and the defect-review deadline, and explain "
    "shipping, refund timing, and what remains unverified. Search and cite both the "
    "Northline Returns Policy and the Aurora H7 No-Audio Support Bulletin. "
    "Write the source titles in plain text after the claims they support.",
    conversation_id=customer_conversation,
    require_file_search=True,
)


## 8. Question 3 — draft a safe customer reply

This continues the same conversation. The agent should offer the remaining checks and a conditional support-review path without claiming the return or refund has already been approved.


In [ ]:
customer_conversation = ask_service_agent(
    "Draft a concise reply to the customer. Explain the likely next step, the "
    "balance/firmware/second-device checks, and what support can do if the issue "
    "persists. Be accurate about the return window, prepaid label, and refund: "
    "do not promise an approved RMA or an immediate refund.",
    conversation_id=customer_conversation,
    require_file_search=True,
)


## 9. Inspect the saved conversation

This reads the conversation through the authenticated SDK. The portal Traces view requires Application Insights and may take a few minutes to populate. The saved conversation can be inspected independently of tracing.


In [ ]:
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
agent_client = None
try:
    agent_client = project.get_openai_client(agent_name=agent.name)
    items = list(agent_client.conversations.items.list(customer_conversation, order="asc"))
    print(f"Saved conversation: {customer_conversation} ({len(items)} items)")
    for number, item in enumerate(items, start=1):
        print(f"\n--- Item {number}: {item.type} ---")
        print(json.dumps(item.model_dump(mode="json", exclude_none=True), indent=2, ensure_ascii=False))
finally:
    if agent_client is not None:
        agent_client.close()
    project.close()


## Expected reasoning and demo boundaries

- Visual document: order NA-28471, Aurora H7 (SKU AH7-BLK), delivered 2026-09-18, request dated 2026-10-21. The customer's complaint and reset attempt are self-reported.
- Arithmetic: 33 calendar days from delivery to case review; the ordinary 30-day return window has passed. The suspected-defect review runs to 2026-11-02, 45 days after delivery.
- Policy: opened headphones can enter defect review; support confirmation and an RMA are needed. If approved, shipping is prepaid, no restocking fee applies, and refund follows receipt and verification to the original payment method.
- Support bulletin: check charge and audio balance, reset/re-pair, update to firmware 2.4.1 or later, and test a second source device before escalation. The image says a reset was tried but does not establish the other checks.
- Evidence boundary: the order image supplies transaction and reported-case facts; the two text documents supply policy and technical procedure. This is a synthetic example and no actual customer transaction is performed.

For production, replace the synthetic documents, add human review and secure order-system integration, and instrument local function execution if you need end-to-end telemetry beyond Foundry's server-side traces.
